# SQL Analytics Layer — Telco Customer Churn
**Input:** data/silver/telco_silver.parquet  
**Output:** data/telco.db (SQLite database) + sql/churn_queries.sql  
**Goal:** Answer 10 real business questions using SQL

In [14]:
import pandas as pd
import sqlite3
import os

# Step 1: Load the cleaned Silver layer
df_silver = pd.read_parquet('C:/Users/DELL/Documents/telecom-churn-analytics/data/silver/telco_silver.parquet')
print(f"✓ Silver data loaded: {df_silver.shape}")

# Step 2: Create a connection to SQLite database
# This single line creates the .db file if it doesn't exist
conn = sqlite3.connect('C:/Users/DELL/Documents/telecom-churn-analytics/data/telco.db')
print(f"✓ Database connection created")

# Step 3: Write the dataframe into the database as a table
# if_exists='replace' means: if the table already exists, overwrite it
df_silver.to_sql('customers', conn, if_exists='replace', index=False)
print(f"✓ Data written to 'customers' table in database")

# Step 4: Verify by reading back a quick count
cursor = conn.cursor()
cursor.execute("SELECT COUNT(*) FROM customers")
row_count = cursor.fetchone()[0]
print(f"✓ Rows in database: {row_count:,}")

✓ Silver data loaded: (7043, 21)
✓ Database connection created
✓ Data written to 'customers' table in database
✓ Rows in database: 7,043


In [15]:
def run_query(sql, description=""):
    """
    Runs a SQL query and returns the result as a pandas dataframe.
    Also prints a header so output is easy to read.
    """
    if description:
        print(f"\n{'='*50}")
        print(f" {description}")
        print(f"{'='*50}")
    
    result = pd.read_sql(sql, conn)
    return result

In [16]:
# Test the helper function
test = run_query("SELECT COUNT(*) as total_rows FROM customers", 
                 "Test Query")
print(test)


 Test Query
   total_rows
0        7043


## Queries 1-4: Foundation Metrics
These are the first questions any telecom manager would ask.
Basic aggregations using SELECT, COUNT, SUM, AVG, GROUP BY.

In [17]:
q1 = run_query("""
    SELECT 
        COUNT(*)                                    AS total_customers,
        SUM(Churn)                                  AS total_churned,
        ROUND(100.0 * SUM(Churn) / COUNT(*), 2)    AS churn_rate_pct
    FROM customers
""", "Q1: Overall Churn Rate")

print(q1)


 Q1: Overall Churn Rate
   total_customers  total_churned  churn_rate_pct
0             7043           1869           26.54


In [18]:
q2 = run_query("""
    SELECT 
        Contract,
        COUNT(*)                                    AS total_customers,
        SUM(Churn)                                  AS churned,
        ROUND(100.0 * SUM(Churn) / COUNT(*), 2)    AS churn_rate_pct
    FROM customers
    GROUP BY Contract
    ORDER BY churn_rate_pct DESC
""", "Q2: Churn Rate by Contract Type")

print(q2)


 Q2: Churn Rate by Contract Type
         Contract  total_customers  churned  churn_rate_pct
0  Month-to-month             3875     1655           42.71
1        One year             1473      166           11.27
2        Two year             1695       48            2.83


In [19]:
q3 = run_query("""
    SELECT 
        SUM(Churn)                              AS churned_customers,
        ROUND(AVG(MonthlyCharges), 2)           AS avg_monthly_charge,
        ROUND(SUM(MonthlyCharges), 2)           AS total_monthly_revenue_at_risk
    FROM customers
    WHERE Churn = 1
""", "Q3: Monthly Revenue at Risk")

print(q3)


 Q3: Monthly Revenue at Risk
   churned_customers  avg_monthly_charge  total_monthly_revenue_at_risk
0               1869               74.44                      139130.85


In [20]:
q4 = run_query("""
    SELECT 
        CASE 
            WHEN tenure BETWEEN 0  AND 12 THEN '1. First Year (0-12 months)'
            WHEN tenure BETWEEN 13 AND 24 THEN '2. Second Year (13-24 months)'
            WHEN tenure BETWEEN 25 AND 48 THEN '3. Years 2-4 (25-48 months)'
            ELSE                                '4. Loyal (49+ months)'
        END                                         AS tenure_band,
        COUNT(*)                                    AS customers,
        SUM(Churn)                                  AS churned,
        ROUND(100.0 * SUM(Churn) / COUNT(*), 2)    AS churn_rate_pct
    FROM customers
    GROUP BY tenure_band
    ORDER BY tenure_band
""", "Q4: Churn Rate by Tenure Band")

print(q4)


 Q4: Churn Rate by Tenure Band
                     tenure_band  customers  churned  churn_rate_pct
0    1. First Year (0-12 months)       2186     1037           47.44
1  2. Second Year (13-24 months)       1024      294           28.71
2    3. Years 2-4 (25-48 months)       1594      325           20.39
3          4. Loyal (49+ months)       2239      213            9.51


## Queries 5-7: Segment Analysis
Deeper business questions about specific customer groups.
Introduces ROUND, AVG, and multi-column GROUP BY.

In [21]:
q5 = run_query("""
    SELECT
        CASE WHEN SeniorCitizen = 1 THEN 'Senior' 
             ELSE 'Non-Senior' 
        END                                         AS customer_type,
        COUNT(*)                                    AS customers,
        SUM(Churn)                                  AS churned,
        ROUND(100.0 * SUM(Churn) / COUNT(*), 2)    AS churn_rate_pct,
        ROUND(AVG(MonthlyCharges), 2)               AS avg_monthly_charge
    FROM customers
    GROUP BY SeniorCitizen
    ORDER BY churn_rate_pct DESC
""", "Q5: Senior vs Non-Senior Churn")

print(q5)


 Q5: Senior vs Non-Senior Churn
  customer_type  customers  churned  churn_rate_pct  avg_monthly_charge
0        Senior       1142      476           41.68               79.82
1    Non-Senior       5901     1393           23.61               61.85


In [22]:
q6 = run_query("""
    SELECT
        PaymentMethod,
        COUNT(*)                                    AS customers,
        SUM(Churn)                                  AS churned,
        ROUND(100.0 * SUM(Churn) / COUNT(*), 2)    AS churn_rate_pct
    FROM customers
    GROUP BY PaymentMethod
    ORDER BY churn_rate_pct DESC
""", "Q6: Churn by Payment Method")

print(q6)


 Q6: Churn by Payment Method
               PaymentMethod  customers  churned  churn_rate_pct
0           Electronic check       2365     1071           45.29
1               Mailed check       1612      308           19.11
2  Bank transfer (automatic)       1544      258           16.71
3    Credit card (automatic)       1522      232           15.24


In [23]:
q7 = run_query("""
    SELECT
        customerID,
        tenure,
        Contract,
        ROUND(MonthlyCharges, 2)    AS monthly_charges,
        ROUND(TotalCharges, 2)      AS total_charges
    FROM customers
    WHERE Churn = 1
      AND MonthlyCharges > 70
      AND Contract = 'Month-to-month'
    ORDER BY MonthlyCharges DESC
    LIMIT 15
""", "Q7: Top 15 High-Value Churned Customers")

print(q7)


 Q7: Top 15 High-Value Churned Customers
    customerID  tenure        Contract  monthly_charges  total_charges
0   2302-ANTDP      48  Month-to-month           117.45        5438.90
1   4361-BKAXE      41  Month-to-month           114.50        4527.45
2   9158-VCTQB      41  Month-to-month           113.60        4594.95
3   7279-BUYWN      41  Month-to-month           113.20        4689.50
4   1583-IHQZE      12  Month-to-month           112.95        1384.75
5   0115-TFERT      21  Month-to-month           111.20        2317.10
6   9079-YEXQJ      54  Month-to-month           111.10        6014.85
7   5099-BAILX      43  Month-to-month           110.75        4687.90
8   3336-JORSO      33  Month-to-month           110.45        3655.45
9   9851-KIELU      10  Month-to-month           110.10        1043.30
10  3992-YWPKO       6  Month-to-month           109.90         669.45
11  3331-HQDTW      34  Month-to-month           109.80        3587.25
12  6599-RCLCJ      47  Month-to-mo

## Queries 8-10: Advanced SQL
Window functions, multi-condition filtering, and CTEs.
These are the queries that come up in data analyst interviews.

In [24]:
q8 = run_query("""
    SELECT
        (PhoneService + OnlineSecurity + OnlineBackup +
         DeviceProtection + TechSupport + 
         StreamingTV + StreamingMovies)              AS services_subscribed,
        COUNT(*)                                     AS customers,
        SUM(Churn)                                   AS churned,
        ROUND(100.0 * SUM(Churn) / COUNT(*), 2)     AS churn_rate_pct
    FROM customers
    GROUP BY services_subscribed
    ORDER BY services_subscribed
""", "Q8: Bundle Effect — Services Count vs Churn Rate")

print(q8)


 Q8: Bundle Effect — Services Count vs Churn Rate
   services_subscribed  customers  churned  churn_rate_pct
0                    0         80       35           43.75
1                    1       2253      488           21.66
2                    2        996      433           43.47
3                    3       1041      361           34.68
4                    4       1062      289           27.21
5                    5        827      182           22.01
6                    6        525       66           12.57
7                    7        259       15            5.79


In [26]:
q9 = run_query("""
    SELECT
        tenure,
        COUNT(*)                                        AS customers_at_tenure,
        SUM(Churn)                                      AS churned_at_tenure,
        SUM(SUM(Churn)) OVER (ORDER BY tenure)          AS running_total_churned,
        ROUND(100.0 * SUM(Churn) / COUNT(*), 2)        AS churn_rate_pct
    FROM customers
    GROUP BY tenure
    ORDER BY tenure
    LIMIT 15
""", "Q9: Running Churn Total by Tenure Month (first 15 months)")

print(q9)


 Q9: Running Churn Total by Tenure Month (first 15 months)
    tenure  customers_at_tenure  churned_at_tenure  running_total_churned  \
0        0                   11                  0                      0   
1        1                  613                380                    380   
2        2                  238                123                    503   
3        3                  200                 94                    597   
4        4                  176                 83                    680   
5        5                  133                 64                    744   
6        6                  110                 40                    784   
7        7                  131                 51                    835   
8        8                  123                 42                    877   
9        9                  119                 46                    923   
10      10                  116                 45                    968   
11      11      

In [27]:
q10 = run_query("""
    WITH high_risk_customers AS (
        SELECT 
            customerID,
            tenure,
            MonthlyCharges,
            Contract,
            Churn
        FROM customers
        WHERE Contract    = 'Month-to-month'
          AND tenure      < 12
          AND MonthlyCharges > 65
    )
    SELECT
        COUNT(*)                                        AS high_risk_total,
        SUM(Churn)                                      AS already_churned,
        COUNT(*) - SUM(Churn)                          AS still_active_at_risk,
        ROUND(100.0 * SUM(Churn) / COUNT(*), 2)        AS churn_rate_pct,
        ROUND(AVG(MonthlyCharges), 2)                  AS avg_monthly_charge,
        ROUND(SUM(MonthlyCharges), 2)                  AS monthly_revenue_at_risk
    FROM high_risk_customers
""", "Q10: CTE — High-Risk Segment Analysis")

print(q10)


 Q10: CTE — High-Risk Segment Analysis
   high_risk_total  already_churned  still_active_at_risk  churn_rate_pct  \
0              931              635                   296           68.21   

   avg_monthly_charge  monthly_revenue_at_risk  
0               81.08                  75484.4  


In [28]:
sql_content = """
-- ============================================================
-- Telecom Customer Churn Analytics — SQL Query Library
-- Author: Ujjawal Kumar
-- Dataset: IBM Telco Customer Churn (Kaggle)
-- Database: SQLite (data/telco.db)
-- ============================================================

-- Q1: Overall churn rate
SELECT 
    COUNT(*)                                    AS total_customers,
    SUM(Churn)                                  AS total_churned,
    ROUND(100.0 * SUM(Churn) / COUNT(*), 2)    AS churn_rate_pct
FROM customers;

-- Q2: Churn rate by contract type (strongest business signal)
SELECT 
    Contract,
    COUNT(*)                                    AS total_customers,
    SUM(Churn)                                  AS churned,
    ROUND(100.0 * SUM(Churn) / COUNT(*), 2)    AS churn_rate_pct
FROM customers
GROUP BY Contract
ORDER BY churn_rate_pct DESC;

-- Q3: Monthly revenue at risk from churned customers
SELECT 
    SUM(Churn)                              AS churned_customers,
    ROUND(AVG(MonthlyCharges), 2)           AS avg_monthly_charge,
    ROUND(SUM(MonthlyCharges), 2)           AS total_monthly_revenue_at_risk
FROM customers
WHERE Churn = 1;

-- Q4: Churn rate by tenure band (CASE WHEN bucketing)
SELECT 
    CASE 
        WHEN tenure BETWEEN 0  AND 12 THEN '1. First Year (0-12 months)'
        WHEN tenure BETWEEN 13 AND 24 THEN '2. Second Year (13-24 months)'
        WHEN tenure BETWEEN 25 AND 48 THEN '3. Years 2-4 (25-48 months)'
        ELSE                                '4. Loyal (49+ months)'
    END                                         AS tenure_band,
    COUNT(*)                                    AS customers,
    SUM(Churn)                                  AS churned,
    ROUND(100.0 * SUM(Churn) / COUNT(*), 2)    AS churn_rate_pct
FROM customers
GROUP BY tenure_band
ORDER BY tenure_band;

-- Q5: Senior vs Non-Senior churn comparison
SELECT
    CASE WHEN SeniorCitizen = 1 THEN 'Senior' 
         ELSE 'Non-Senior' 
    END                                         AS customer_type,
    COUNT(*)                                    AS customers,
    SUM(Churn)                                  AS churned,
    ROUND(100.0 * SUM(Churn) / COUNT(*), 2)    AS churn_rate_pct,
    ROUND(AVG(MonthlyCharges), 2)               AS avg_monthly_charge
FROM customers
GROUP BY SeniorCitizen
ORDER BY churn_rate_pct DESC;

-- Q6: Churn by payment method
SELECT
    PaymentMethod,
    COUNT(*)                                    AS customers,
    SUM(Churn)                                  AS churned,
    ROUND(100.0 * SUM(Churn) / COUNT(*), 2)    AS churn_rate_pct
FROM customers
GROUP BY PaymentMethod
ORDER BY churn_rate_pct DESC;

-- Q7: Top high-value churned customers for CRM targeting
SELECT
    customerID,
    tenure,
    Contract,
    ROUND(MonthlyCharges, 2)    AS monthly_charges,
    ROUND(TotalCharges, 2)      AS total_charges
FROM customers
WHERE Churn = 1
  AND MonthlyCharges > 70
  AND Contract = 'Month-to-month'
ORDER BY MonthlyCharges DESC
LIMIT 15;

-- Q8: Bundle effect — services count vs churn rate
SELECT
    (PhoneService + OnlineSecurity + OnlineBackup +
     DeviceProtection + TechSupport + 
     StreamingTV + StreamingMovies)             AS services_subscribed,
    COUNT(*)                                    AS customers,
    SUM(Churn)                                  AS churned,
    ROUND(100.0 * SUM(Churn) / COUNT(*), 2)    AS churn_rate_pct
FROM customers
GROUP BY services_subscribed
ORDER BY services_subscribed;

-- Q9: Window function — running churn total by tenure month
SELECT
    tenure,
    COUNT(*)                                        AS customers_at_tenure,
    SUM(Churn)                                      AS churned_at_tenure,
    SUM(SUM(Churn)) OVER (ORDER BY tenure)          AS running_total_churned,
    ROUND(100.0 * SUM(Churn) / COUNT(*), 2)        AS churn_rate_pct
FROM customers
GROUP BY tenure
ORDER BY tenure;

-- Q10: CTE — High-risk customer segment definition and sizing
WITH high_risk_customers AS (
    SELECT 
        customerID, tenure, MonthlyCharges, Contract, Churn
    FROM customers
    WHERE Contract      = 'Month-to-month'
      AND tenure        < 12
      AND MonthlyCharges > 65
)
SELECT
    COUNT(*)                                        AS high_risk_total,
    SUM(Churn)                                      AS already_churned,
    COUNT(*) - SUM(Churn)                          AS still_active_at_risk,
    ROUND(100.0 * SUM(Churn) / COUNT(*), 2)        AS churn_rate_pct,
    ROUND(AVG(MonthlyCharges), 2)                  AS avg_monthly_charge,
    ROUND(SUM(MonthlyCharges), 2)                  AS monthly_revenue_at_risk
FROM high_risk_customers;
"""

# Write to the sql/ folder
with open('C:/Users/DELL/Documents/telecom-churn-analytics/sql/churn_queries.sql', 'w') as f:
    f.write(sql_content)

print("✓ All 10 queries saved to sql/churn_queries.sql")
print(f"  File size: {os.path.getsize('C:/Users/DELL/Documents/telecom-churn-analytics/sql/churn_queries.sql')} bytes")

✓ All 10 queries saved to sql/churn_queries.sql
  File size: 4832 bytes


In [29]:
# Always close the connection when done
conn.close()
print("✓ Database connection closed")
print("\nDay 2 SQL layer complete!")
print("  ✓ telco.db created with customers table")
print("  ✓ 10 business queries written and tested")
print("  ✓ churn_queries.sql saved to sql/ folder")

✓ Database connection closed

Day 2 SQL layer complete!
  ✓ telco.db created with customers table
  ✓ 10 business queries written and tested
  ✓ churn_queries.sql saved to sql/ folder
